In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data import (
    ARTIFACTS_DIR,
    CLEANED_DIR,
    calculate_pearson_correlations,
    load_and_preprocess_system,
    resolve_system_file_path,
    run_feature_selection_pipeline,
)

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print(f"Cleaned Data Directory: {CLEANED_DIR}")

In [ ]:
sys4_path = resolve_system_file_path(CLEANED_DIR, "system_4")
df_sys4 = load_and_preprocess_system(sys4_path)

print(f"Loaded file: {sys4_path.name}")
print(f"Daytime production rows (dc_power > 0): {len(df_sys4):,}")
df_sys4.describe().T[["count", "mean", "std", "min", "50%", "max"]]

In [ ]:
corr_series = calculate_pearson_correlations(df_sys4)
print("Pearson Correlation Coefficients (vs. dc_power):")
print(corr_series)

# Heatmap plot
plt.figure(figsize=(8, 6))
corr_matrix = df_sys4.select_dtypes(include=["number"]).corr()
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1
)
plt.title("System 4 — Feature Correlation Matrix")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "system4_correlation_heatmap.png", dpi=300)
plt.show()

In [ ]:
numeric_cols = [
    c for c in df_sys4.select_dtypes(include=["number"]).columns if c != "dc_power"
]
n_cols = 2
n_rows = (len(numeric_cols) + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    axes[idx].scatter(df_sys4[col], df_sys4["dc_power"], alpha=0.1, s=1)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("dc_power")
    axes[idx].set_title(f"{col} vs dc_power (r = {corr_series.loc[col]:.2f})")

for idx in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "system4_scatter_plots.png", dpi=300)
plt.show()

In [ ]:
selected_feats, primary_corr, val_df = run_feature_selection_pipeline(
    cleaned_dir=CLEANED_DIR,
    artifacts_dir=ARTIFACTS_DIR,
    primary_system="system_4",
    validation_systems=["system_10", "system_33", "system_50"],
    threshold=0.3,
)

print(f"\nFinal Selected Features (|r| >= 0.3): {selected_feats}")
print("\nValidation across Systems (Pearson r):")
display(val_df)

In [ ]:
# Cell 6: Global Feature Selection via Hardware Grouping
import os
from src.data import run_group_feature_selection

# Map systems to groups based on identical hardware sensors
SYSTEM_GROUPS = {
    "group_34_35": ["system_34", "system_35"],                 
    "group_50_51": ["system_50", "system_51"],                 
    "group_1276_1277": ["system_1276", "system_1277"],         
    "group_base_4_1283_1289": ["system_4", "system_1283", "system_1289"], 
    "group_10": ["system_10"],                                 
    "group_33": ["system_33"],                                 
    "group_36": ["system_36"],                                 
    "group_1200": ["system_1200"]                              
}

print("Executing group-level feature selection (this may take a moment for large datasets)...")
group_features = run_group_feature_selection(
    cleaned_dir=CLEANED_DIR,
    artifacts_dir=ARTIFACTS_DIR,
    system_groups=SYSTEM_GROUPS,
    threshold=0.3
)

# Output Verification
for group, features in group_features.items():
    print(f"\n✅ {group} Selected Features (|r| >= 0.3):")
    print(features)

print(f"\nVisual Artifacts generated in {ARTIFACTS_DIR}:")
print([f for f in os.listdir(ARTIFACTS_DIR) if f.endswith('.png') and 'group' in f])

# Advanced Feature Selection with Hyperparameters testing
### Non-Linearity & Multicollinearity Pruning



METRIC: Determines whether the algorithm isolates strict linear relationships (pearson) or flexible, monotonic non-linear curves (spearman)
TARGET_THRESHOLD: Sets the minimum correlation score against $P_{DC}$ required to retain a feature, controlling how aggressively the dataset is pruned.
COLLINEARITY_LIMIT: Eliminates redundant sensors by dropping features that correlate with each other above this threshold, minimizing tree model computation time.
IRRADIANCE_MIN: Masks transient dawn/dusk sensor noise by evaluating correlations only when physical irradiance ($W/m^2$) exceeds this baseline.

In [ ]:
# Cell 7: Safe Feature Selection Experimentation Sandbox
from src.data import run_experimental_feature_selection, CLEANED_DIR, ARTIFACTS_DIR

# Create isolated directory for experiments
EXP_DIR = ARTIFACTS_DIR / "experiments"
EXP_DIR.mkdir(exist_ok=True)

# Re-declare groups to ensure memory scope safety
SYSTEM_GROUPS = {
    "group_34_35": ["system_34", "system_35"],                 
    "group_50_51": ["system_50", "system_51"],                 
    "group_1276_1277": ["system_1276", "system_1277"],         
    "group_base_4_1283_1289": ["system_4", "system_1283", "system_1289"], 
    "group_10": ["system_10"],                                 
    "group_33": ["system_33"],                                 
    "group_36": ["system_36"],                                 
    "group_1200": ["system_1200"]                              
}

# --- TUNING DIALS ---
METRIC = "spearman" 
TARGET_THRESHOLD = 0.10 
COLLINEARITY_LIMIT = .9
IRRADIANCE_MIN = 10.0 
# --------------------

print(f"Running Experiment: {METRIC.upper()}, Target \u2265 {TARGET_THRESHOLD}, Collinearity \u2264 {COLLINEARITY_LIMIT}")
experimental_features = run_experimental_feature_selection(
    cleaned_dir=CLEANED_DIR,
    artifacts_dir=EXP_DIR,  # <-- SAVING TO ISOLATED FOLDER
    system_groups=SYSTEM_GROUPS,
    target_corr_threshold=TARGET_THRESHOLD,
    collinearity_threshold=COLLINEARITY_LIMIT,
    corr_method=METRIC,
    irradiance_threshold=IRRADIANCE_MIN
)

for group, features in experimental_features.items():
    print(f"[{group}] Pruned Features: {features}")

In [ ]:
import pandas as pd
from pathlib import Path

def analyze_feature_experiments_with_features(exp_dir: Path) -> pd.DataFrame:
    """Parses experimental registries and exports full feature lists to CSV."""
    data = []
    
    for file in exp_dir.glob("*.txt"):
        # Safely split filename to extract group and parameters
        parts = file.stem.split("_exp_")
        if len(parts) != 2: 
            continue
        
        group_name = parts[0].replace("selected_features_", "")
        exp_params = parts[1]
        
        # Read selected variables
        with open(file, "r") as f:
            features = [line.strip() for line in f.readlines() if line.strip()]
            
        data.append({
            "Group": group_name,
            "Experiment_Params": exp_params,
            "Feature_Count": len(features),
            "Selected_Features": ", ".join(features)  # Retaining the full list
        })
        
    df = pd.DataFrame(data).sort_values(by=["Group", "Experiment_Params"])
    
    # Create a display copy to prevent console wrapping issues
    display_df = df.copy()
    display_df["Selected_Features"] = display_df["Selected_Features"].apply(
        lambda x: (x[:70] + "...") if len(x) > 70 else x
    )
    print(display_df.to_markdown(index=False))
    
    return df

# Execute and export
EXP_DIR = Path("artifacts/experiments")
if EXP_DIR.exists():
    df_analysis = analyze_feature_experiments_with_features(EXP_DIR)
    
    csv_path = EXP_DIR / "experiment_summary_with_features.csv"
    df_analysis.to_csv(csv_path, index=False)
    print(f"\n Full summary with selected features exported to: {csv_path}")
else:
    print(f"Error: Directory {EXP_DIR} not found.")

Following an experimental hyperparameter sweep of the feature selection pipeline, a strict linear configuration (pearson, $\tau = 0.3$, $\rho_{max} = 0.8$, $POA \ge 20 \ W/m^2$) was identified as optimal for the baseline regression architecture.Irradiance Masking ($POA \ge 20 \ W/m^2$): Enforcing a physical irradiance threshold effectively dropped non-linear transient noise generated during inverter start-up and shut-down phases (dawn/dusk), ensuring the covariance matrix was fitted purely to steady-state generation.Collinearity Pruning ($\rho_{max} \le 0.8$): The multicollinearity filter successfully eliminated redundant thermodynamic and electrical sensors. For example, it systematically dropped ambient_temp in favor of module_temp, as both exhibit $r > 0.85$ collinearity, thereby reducing dimensionality without sacrificing explained variance.Target Threshold ($\tau \ge 0.3$): This boundary mathematically justified the exclusion of weak environmental signals (such as wind_speed and humidity found in systems 34, 35, 1276, and 1277), ensuring the models are not penalized by noise from variables lacking strong linear predictive power for instantaneous $P_{DC}$."   